This script makes docker image, pushes to ECR, and makes a Sagemaker Endpoint instance(s)

In [1]:
#SET VARIABLES FOR PROJECT HERE

import time
import os
import json

#SET VARIABLES HERE for ECR, model, endpoint, project name
ecr_repository = "[REDACTED]-ml/math-proficiency-model"
endpoint_name = 'math-proficiency-endpoint'
instance_type = 'ml.c5.large'

#aws account vairables
account_id = '[REDACTED]'
region = 'us-east-1'

#test message (for inference)
input_data = [
        {"past_performance": [1.0,1.0,1,0], "skill_optimal_threshold": 0.5, "item_optimal_threshold": 0.5},
        {"past_performance": [0.0,1.0,0,0], "skill_optimal_threshold": 1.0, "item_optimal_threshold": 1.0},
    ]
test_message = json.dumps({
        "inputs": input_data
    })


# test_message = json.dumps({
#     "past_performance": [[1.0, 5.0],[1.0, 1.0]]
# })

#scaling parameters for server
min_capacity = 1
max_capacity = 5
target_value = 1000  # Target invocations per instance per minute
scale_in_cooldown = 300  # 5 minutes
scale_out_cooldown = 300  # 5 minutes


#subdirectory model + artifacts live in//current folder name
import os
project_name = os.path.basename(os.getcwd())

# Set variables for Sagemaker Endpoints
image_tag = str(round(time.time()*1000))

#copy variables for docker scripts
AWS_ACCOUNT_ID = account_id
REGION = region
IMAGE_TAG = image_tag
ECR_REPOSITORY = ecr_repository


# Construct the ECR image URI
image_uri = f'{account_id}.dkr.ecr.{region}.amazonaws.com/{ecr_repository}:{image_tag}'

print('project name: ', project_name, '\nECR Image URI: ', image_uri)

project name:  student_proficiency_model 
ECR Image URI:  [REDACTED].dkr.ecr.us-east-1.amazonaws.com/[REDACTED]-ml/math-proficiency-model:1722474790779


In [2]:
#buid docker, push to ECR

import subprocess

def run_command(command):
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE, shell=True)
    output, error = process.communicate()
    if process.returncode != 0:
        print(f"Error executing command: {command}")
        print(f"Error message: {error.decode('utf-8')}")
        raise Exception(f"Command failed with return code {process.returncode}")
    return output.decode('utf-8')

#look at dockerfile
dockerfile_path = os.path.join('container', 'Dockerfile')
with open(dockerfile_path, 'r') as file:
    dockerfile_contents = file.read()
print("Contents of container/Dockerfile:")
print(dockerfile_contents)

# Change directory to 'container'
os.chdir('container')

# Authenticate Docker to AWS ECR
run_command(f"aws ecr get-login-password --region {REGION} | docker login --username AWS --password-stdin 763104351884.dkr.ecr.{REGION}.amazonaws.com")

# Also authenticate to your own ECR repository
run_command(f"aws ecr get-login-password --region {REGION} | docker login --username AWS --password-stdin {AWS_ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com")

# Create ECR repository if it doesn't exist
try:
    run_command(f"aws ecr describe-repositories --repository-names {ECR_REPOSITORY}")
except Exception:
    run_command(f"aws ecr create-repository --repository-name {ECR_REPOSITORY}")

# Build the Docker image
run_command(f"docker build -t {ECR_REPOSITORY}:{IMAGE_TAG} .")

# Tag the image for ECR
run_command(f"docker tag {ECR_REPOSITORY}:{IMAGE_TAG} {AWS_ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com/{ECR_REPOSITORY}:{IMAGE_TAG}")

# Push the image to ECR
run_command(f"docker push {AWS_ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com/{ECR_REPOSITORY}:{IMAGE_TAG}")

print(f"Image pushed successfully to ECR: {AWS_ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com/{ECR_REPOSITORY}:{IMAGE_TAG}")

Contents of container/Dockerfile:
# Start with the AWS Deep Learning Container for Python 3.8
#FROM 763104351884.dkr.ecr.us-east-1.amazonaws.com/pytorch-inference:1.8.1-cpu-py38-ubuntu20.04
# FROM pytorch/pytorch:1.8.1-cuda11.1-cudnn8-runtime
FROM 763104351884.dkr.ecr.us-east-1.amazonaws.com/pytorch-inference:1.12.1-cpu-py38-ubuntu20.04-sagemaker

# Set working directory
WORKDIR /opt/ml/model

# Install the required packages
RUN pip install --no-cache-dir --upgrade pip && \
    pip install --no-cache-dir \
    numpy \
    scipy \
    scikit-learn==1.3.2 \
    pandas \
    flask \
    gunicorn \
    pickle5

# Copy your model file and any other necessary files
COPY model.pkl /opt/ml/model/
COPY inference.py /opt/ml/model/

# Set the entrypoint to gunicorn
ENTRYPOINT ["gunicorn", "--bind", "0.0.0.0:8080", "inference:app"]
Image pushed successfully to ECR: [REDACTED].dkr.ecr.us-east-1.amazonaws.com/[REDACTED]-ml/math-proficiency-model:1722474790779


In [3]:
#Boto3/Sagemaker update interaction functions
#Updating, scaling, deregister scaling, health checks, rollbacks

import boto3
import time
import sagemaker
from sagemaker import get_execution_role
import json


def deregister_scalable_target(endpoint_name):
    client = boto3.client('application-autoscaling')
    resource_id = f'endpoint/{endpoint_name}/variant/AllTraffic'
    
    try:
        client.deregister_scalable_target(
            ServiceNamespace='sagemaker',
            ResourceId=resource_id,
            ScalableDimension='sagemaker:variant:DesiredInstanceCount'
        )
        print(f"Deregistered scalable target for endpoint: {endpoint_name}")
    except client.exceptions.ObjectNotFoundException:
        print(f"Scalable target for endpoint {endpoint_name} not found. Proceeding with update.")


def register_scalable_target(endpoint_name, min_capacity, max_capacity):
    client = boto3.client('application-autoscaling')
    resource_id = f'endpoint/{endpoint_name}/variant/AllTraffic'
    
    client.register_scalable_target(
        ServiceNamespace='sagemaker',
        ResourceId=resource_id,
        ScalableDimension='sagemaker:variant:DesiredInstanceCount',
        MinCapacity=min_capacity,
        MaxCapacity=max_capacity
    )
    print(f"Registered scalable target for endpoint: {endpoint_name}")

#
#this could be redundant to the register_scalable_target() but sets more params for scaling    
def setup_auto_scaling(endpoint_name, min_capacity, max_capacity, target_value, 
                       scale_in_cooldown=300, scale_out_cooldown=300):
    client = boto3.client('application-autoscaling')
    
    # Register the endpoint as a scalable target
    client.register_scalable_target(
        ServiceNamespace='sagemaker',
        ResourceId=f'endpoint/{endpoint_name}/variant/AllTraffic',
        ScalableDimension='sagemaker:variant:DesiredInstanceCount',
        MinCapacity=min_capacity,
        MaxCapacity=max_capacity
    )
    
    # Define and apply scaling policy
    client.put_scaling_policy(
        PolicyName=f'ScalingPolicy-{endpoint_name}',
        ServiceNamespace='sagemaker',
        ResourceId=f'endpoint/{endpoint_name}/variant/AllTraffic',
        ScalableDimension='sagemaker:variant:DesiredInstanceCount',
        PolicyType='TargetTrackingScaling',
        TargetTrackingScalingPolicyConfiguration={
            'TargetValue': target_value,
            'PredefinedMetricSpecification': {
                'PredefinedMetricType': 'SageMakerVariantInvocationsPerInstance'
            },
            'ScaleInCooldown': scale_in_cooldown,
            'ScaleOutCooldown': scale_out_cooldown
        }
    )

    print(f"Auto-scaling has been set up for endpoint: {endpoint_name}")
    print(f"Min capacity: {min_capacity}")
    print(f"Max capacity: {max_capacity}")
    print(f"Target value: {target_value} invocations per instance per minute")
    print(f"Scale-in cooldown: {scale_in_cooldown} seconds")
    print(f"Scale-out cooldown: {scale_out_cooldown} seconds")
    
def update_endpoint(endpoint_name, new_image_uri, test_message):
    sagemaker_client = boto3.client('sagemaker')

    # Get the current endpoint configuration
    current_endpoint_config = sagemaker_client.describe_endpoint(EndpointName=endpoint_name)['EndpointConfigName']

    # Get the current model name and image
    current_config = sagemaker_client.describe_endpoint_config(EndpointConfigName=current_endpoint_config)
    current_model_name = current_config['ProductionVariants'][0]['ModelName']
    current_model = sagemaker_client.describe_model(ModelName=current_model_name)
    current_image_uri = current_model['PrimaryContainer']['Image']

    try:
        # Create a new model with the new image
        
        #update model name w current datetime, eg 'math-proficiency-model-2024-08-01-00-54-25-247'
        import re
        from datetime import datetime
        # Extract the base name (everything before the datetime)
        base_model_name = re.split(r'-\d{4}-', current_model_name)[0]
        new_model_name = '-'.join([base_model_name, datetime.now().strftime('%Y-%m-%d-%H-%M-%S-%f')[:-3]])

        sagemaker_client.create_model(
            ModelName=new_model_name,
            PrimaryContainer={
                'Image': new_image_uri,
            },
            ExecutionRoleArn=get_execution_role()
        )

        # Create a new endpoint configuration
        new_config_name = f"{current_endpoint_config}-{new_image_uri.split(':', 1)[-1]}"
        sagemaker_client.create_endpoint_config(
            EndpointConfigName=new_config_name,
            ProductionVariants=[{
                'InstanceType': current_config['ProductionVariants'][0]['InstanceType'],
                'InitialInstanceCount': current_config['ProductionVariants'][0]['InitialInstanceCount'],
                'ModelName': new_model_name,
                'VariantName': 'AllTraffic'
            }]
        )
        print("New config name: ", new_config_name, " Old config name: ", current_endpoint_config)

        # Update the endpoint
        sagemaker_client.update_endpoint(
            EndpointName=endpoint_name,
            EndpointConfigName=new_config_name
        )

        print(f"Endpoint '{endpoint_name}' is being updated. This may take a few minutes...")
        waiter = sagemaker_client.get_waiter('endpoint_in_service')
        waiter.wait(EndpointName=endpoint_name)

        # Check if the new endpoint is healthy
        if check_endpoint_health(endpoint_name, test_message):
            print(f"Endpoint '{endpoint_name}' has been successfully updated with the new model.")
        else:
            raise Exception("New endpoint failed health check")
        
    #if fails, do a rollback
    except Exception as e:
        print(f"Update failed: {str(e)}. Rolling back to previous version...")

        # Rollback to the previous configuration
        sagemaker_client.update_endpoint(
            EndpointName=endpoint_name,
            EndpointConfigName=current_endpoint_config
        )

        print(f"Rolling back endpoint '{endpoint_name}'. This may take a few minutes...")
        waiter = sagemaker_client.get_waiter('endpoint_in_service')
        waiter.wait(EndpointName=endpoint_name)

        if check_endpoint_health(endpoint_name, test_message):
            print(f"Endpoint '{endpoint_name}' has been successfully rolled back to the previous version.")
        else:
            print(f"WARNING: Endpoint '{endpoint_name}' is not healthy after rollback. Manual intervention may be required.")
    
def update_endpoint_with_scaling(endpoint_name, new_image_uri, test_message, min_capacity, max_capacity):
    #You can't update docker image of scalable instance -- so,
    #we need to removing scaling, update, reapply scaling
    try:
        print('Attempting deregistration of scalable target...')
        deregister_scalable_target(endpoint_name)
        print('Attempting updating of endpoint...')
        update_endpoint(endpoint_name, new_image_uri, test_message)
    finally:
        print('Attempting registration of scaling of target...')
        register_scalable_target(endpoint_name, min_capacity, max_capacity)

# Function to check if the endpoint exists
def endpoint_exists(endpoint_name):
    try:
        sagemaker_client.describe_endpoint(EndpointName=endpoint_name)
        return True
    except ClientError as e:
        if e.response['Error']['Code'] == 'ValidationException':
            return False
        else:
            raise e

def check_endpoint_health(endpoint_name, test_message):
    runtime = boto3.client('sagemaker-runtime')
    try:
        # Replace this with an actual inference request that suits your model
        response = runtime.invoke_endpoint(
            EndpointName=endpoint_name,
            ContentType='application/json',
            Body=test_message
        )
        return response['ResponseMetadata']['HTTPStatusCode'] == 200
    except Exception as e:
        print(f"Health check failed: {str(e)}")
        return False            

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


In [4]:
#Deploy if it doesn't exist; otherwise rolling update; rollback if fails

import boto3
import sagemaker
from sagemaker import get_execution_role
from botocore.exceptions import ClientError

# Set up the SageMaker session and role
sagemaker_session = sagemaker.Session()
role = get_execution_role()


# Create the model
model = sagemaker.Model(
    image_uri=image_uri,
    role=role
)

# Initialize the SageMaker client
sagemaker_client = boto3.client('sagemaker')


# Check if the endpoint exists
if not endpoint_exists(endpoint_name):
    # Deploy the model to create an endpoint
    predictor = model.deploy(
        initial_instance_count=1,
        instance_type=instance_type,
        endpoint_name=endpoint_name
    )

    #Create endpoint
    print(f"Endpoint '{endpoint_name}' is being created. This may take a few minutes...")

    # Wait for the endpoint to be in service
    waiter = boto3.client('sagemaker').get_waiter('endpoint_in_service')
    waiter.wait(EndpointName=endpoint_name)

    print(f"Endpoint '{endpoint_name}' is now in service and ready for real-time inference.")
    
    print(f"Attempting to register scaling.")    
    setup_auto_scaling(endpoint_name, min_capacity, max_capacity, target_value, 
                       scale_in_cooldown, scale_out_cooldown)
    
    print(f"Finished.")
else:
    #Endpoint exists, do a rolling update/deploy
    print(f"Endpoint '{endpoint_name}' already exists.")
    print('Attempting update...')
    #this function will descale, update, then rescale
    update_endpoint_with_scaling(endpoint_name, image_uri, test_message, min_capacity, max_capacity)


Endpoint 'math-proficiency-endpoint' already exists.
Attempting update...
Attempting deregistration of scalable target...
Deregistered scalable target for endpoint: math-proficiency-endpoint
Attempting updating of endpoint...
New config name:  math-proficiency-endpoint-1722474790779  Old config name:  math-proficiency-endpoint
Endpoint 'math-proficiency-endpoint' is being updated. This may take a few minutes...
Endpoint 'math-proficiency-endpoint' has been successfully updated with the new model.
Attempting registration of scaling of target...
Registered scalable target for endpoint: math-proficiency-endpoint
